# 04 · Motivation (d) — Shared external stimulus

**Toy example 2.** Every vehicle is subject to the same external disturbance
$g(x) = \alpha\cos(2\pi x)$ on top of its own phase-shifted response:

$$ r_i(x) = \sin\big(2\pi(x+\theta_i)\big) + g(x). $$

Here the global model is *exactly* the right tool for the shared part — it
recovers $g(x)$ — but says nothing about the individual responses. The two-shot
decomposition separates the two: the common component captures $g(x)$, the
personalized component captures each vehicle's phase-shifted response.

Section 2 additionally shows an *alternating-minimization* (iterative) estimator
of the same $C\Psi + P\Phi$ model, as a sanity check that the closed-form
two-shot PCA reaches the same answer.

**Figures produced:** `External_Stimulus.png`, `Stimulus_alternating_min.png`,
`Stimulus_personalized.png`, `Stimulus_common.png`, `Stimulus_unique.png`,
`Stimulus_time_space.png`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "bhd").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.optimize import minimize
from scipy.ndimage import gaussian_filter1d
from sklearn.decomposition import PCA

from bhd import FIG_DIR, set_style

set_style()

## 1. Responses under a shared disturbance, and the global model

In [ ]:
np.random.seed(42)
num_vehicles = 5
x = np.linspace(0, 1, 500)
theta_values = np.random.uniform(0, 1, num_vehicles)
amplitude = 1.0
alpha = 0.5
g = alpha * np.cos(2 * np.pi * x)  # external disturbance shared by all vehicles

responses = [amplitude * np.sin(2 * np.pi * (x + theta)) + g for theta in theta_values]
D = np.column_stack(responses)  # (500, 5)
colors = cm.viridis(np.linspace(0, 1, num_vehicles))


def global_model(x, w):
    """Global model: learn the disturbance magnitude."""
    return w[0] * np.cos(2 * np.pi * x)


def loss_function(w):
    total_loss = 0
    for theta in theta_values:
        true_response = amplitude * np.sin(2 * np.pi * (x + theta)) + g
        total_loss += np.mean((global_model(x, w) - true_response) ** 2)
    return total_loss


result = minimize(loss_function, [0.0], method="BFGS")
optimal_w = result.x
g_hat = global_model(x, optimal_w)
print(f"Optimized disturbance magnitude (w[0]): {optimal_w[0]:.4f}  (true alpha = {alpha})")
print(f"Final loss: {loss_function(optimal_w):.6f}")

plt.figure(figsize=(10, 7))
for i, response in enumerate(responses):
    plt.plot(x, response, color=colors[i], alpha=0.25, linewidth=7)
plt.plot([], [], color="black", linewidth=7, alpha=0.25, label="Vehicle i")
plt.plot(x, g_hat, color="black", linewidth=4, linestyle="-", label="Global Model")
plt.plot(x, g, color="red", linewidth=4, linestyle="-", label="Disturbance g(x)")
plt.xlabel("x", fontsize=20)
plt.ylabel("Response", fontsize=20)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.ylim(-1.85, 1.5)
plt.legend(fontsize=20, loc="lower center", ncol=3)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "External_Stimulus.png", dpi=300, bbox_inches="tight")
plt.show()

## 2. Alternating-minimization estimator (iterative sanity check)

Starting from an SVD initialization, alternate between updating each vehicle's
personalized basis $P_v$ and the common basis $C$ until the reconstruction loss
stops changing.

In [ ]:
np.random.seed(42)
num_common_components = 1
num_personalized_components = 2
n, V = D.shape

U, S_, VT = np.linalg.svd(D, full_matrices=False)
C = U[:, :num_common_components]
P = [np.random.rand(n, num_personalized_components) for _ in range(V)]
Psi = np.zeros((num_common_components, V))
Phi = [np.zeros((num_personalized_components, num_personalized_components)) for _ in range(V)]


def orthogonalize(matrix):
    q, _ = np.linalg.qr(matrix)
    return q


max_iters, threshold = 100, 1e-5
loss_history = []
for iteration in range(max_iters):
    for v in range(V):
        residual = D[:, v].reshape(-1, 1) - C @ Psi[:, v].reshape(-1, 1)
        P[v] = orthogonalize(residual @ residual.T)[:, :num_personalized_components]
        Phi[v] = P[v].T @ D[:, v].reshape(-1, 1)
    residual_sum = sum(D[:, v].reshape(-1, 1) - P[v] @ Phi[v] for v in range(V))
    C = orthogonalize(residual_sum)[:, :num_common_components]
    Psi = C.T @ D
    loss = sum(
        np.linalg.norm(D[:, v].reshape(-1, 1) - (C @ Psi[:, v].reshape(-1, 1) + P[v] @ Phi[v])) ** 2
        for v in range(V)
    )
    loss_history.append(loss)
    if iteration > 0 and abs(loss_history[-1] - loss_history[-2]) < threshold:
        break
print(f"Alternating minimization stopped after {len(loss_history)} iterations, loss = {loss_history[-1]:.4e}")

personalized_predictions = [C @ Psi[:, v].reshape(-1, 1) + P[v] @ Phi[v] for v in range(V)]
smoothed_predictions = [gaussian_filter1d(pred.flatten(), sigma=5) for pred in personalized_predictions]

plt.figure(figsize=(10, 7))
for i, response in enumerate(responses):
    plt.plot(x, response, color=colors[i], alpha=0.25, linewidth=7)
    plt.plot(x, smoothed_predictions[i], linestyle=":", color=colors[i], linewidth=4)
plt.plot([], [], color="black", linewidth=7, alpha=0.25, label="Vehicle i")
plt.plot([], [], color="black", linestyle=":", linewidth=4, label="Personalized Prediction i")
plt.plot(x, g_hat, color="black", linestyle="-", label="Global Model", linewidth=4)
plt.plot(x, g, color="red", linestyle="-", label="Disturbance g(x)", linewidth=4)
plt.xlabel("x", fontsize=20)
plt.ylabel("Response", fontsize=20)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
plt.ylim(-2.1, 1.5)
plt.legend(fontsize=20, loc="lower center", ncol=2)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Stimulus_alternating_min.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(loss_history, color="blue", linewidth=2, label="Reconstruction Loss")
plt.xlabel("Iteration", fontsize=14)
plt.ylabel("Loss", fontsize=14)
plt.title("Loss convergence during alternating minimization", fontsize=16)
plt.legend(fontsize=14)
plt.grid(True)
plt.show()

## 3. Two-shot decomposition (closed form)

In [ ]:
def two_shot_decomposition(D, num_common=1, num_personalized=1):
    n, V = D.shape
    pca_shared = PCA(n_components=num_common)
    C = pca_shared.fit(D.T).components_.T
    Psi = C.T @ D
    D_shared = C @ Psi
    personalized, reconstructed = [], []
    for v in range(V):
        R_v = (D[:, v] - D_shared[:, v]).reshape(-1, 1)
        pca_personal = PCA(n_components=num_personalized)
        Phi_v = pca_personal.fit_transform(R_v)
        P_v = pca_personal.components_
        R_hat = Phi_v @ P_v
        personalized.append(R_hat.flatten())
        reconstructed.append(D_shared[:, v] + R_hat.flatten())
    return D_shared, personalized, reconstructed


D_shared, personalized, reconstructed = two_shot_decomposition(D)
personalized_matrix = np.column_stack(reconstructed)

plt.figure(figsize=(10, 7))
for i in range(num_vehicles):
    plt.plot(x, D[:, i], color=colors[i], alpha=0.25, linewidth=7)
    plt.plot(x, reconstructed[i], linestyle=":", color=colors[i], linewidth=4)
plt.plot([], [], color="black", linewidth=7, alpha=0.25, label="Vehicle i")
plt.plot([], [], color="black", linestyle=":", linewidth=4, alpha=0.25, label="Personalized Prediction i")
plt.plot(x, g_hat, color="black", linewidth=4, linestyle="-", label="Global Model")
plt.plot(x, g, color="red", linewidth=4, linestyle="-", label="Disturbance g(x)")
plt.xlabel("x", fontsize=22)
plt.xticks(fontsize=20)
plt.ylabel("Response", fontsize=22)
plt.yticks([-1.5, -1, -0.5, 0, 0.5, 1, 1.5], fontsize=20)
plt.ylim(-2.2, 1.5)
plt.legend(loc="lower center", ncol=2, fontsize=22)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Stimulus_personalized.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 7))
for i in range(num_vehicles):
    plt.plot(x, D_shared[:, i], color=colors[i], alpha=0.4, linewidth=7)
plt.xlabel("x", fontsize=22)
plt.ylabel("Response", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.ylim(-1.5, 1.5)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Stimulus_common.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 7))
for i in range(num_vehicles):
    plt.plot(x, personalized[i], linestyle="--", color=colors[i], alpha=0.4, linewidth=7)
plt.xlabel("x", fontsize=22)
plt.ylabel("Response", fontsize=22)
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.ylim(-1.5, 1.5)
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_DIR / "Stimulus_unique.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Space–time view: true vs. global vs. personalized

In [ ]:
global_matrix = np.tile(g_hat.reshape(-1, 1), (1, num_vehicles))

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
vmin = min(D.min(), personalized_matrix.min(), global_matrix.min())
vmax = max(D.max(), personalized_matrix.max(), global_matrix.max())
panels = [
    (D, "True Vehicle Responses"),
    (global_matrix, "Global Model Response"),
    (personalized_matrix, "Personalized Model Response"),
]
for ax, (M, title) in zip(axes, panels):
    ax.imshow(M.T, extent=[0, 1, 1, num_vehicles], aspect="auto", cmap="viridis", origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=20)
    ax.set_ylabel("Vehicle Index", fontsize=20)
    ax.set_yticks(np.arange(1, num_vehicles + 1))
    ax.set_yticklabels([str(i) for i in range(1, num_vehicles + 1)])
    ax.tick_params(axis="both", labelsize=18)
axes[2].set_xlabel("x", fontsize=20)
plt.tight_layout()
plt.savefig(FIG_DIR / "Stimulus_time_space.png", dpi=300, bbox_inches="tight")
plt.show()